# 2 Autoencoders

[**Autoencoder (2006):**](https://www.cs.toronto.edu/~hinton/absps/science.pdf) NN used for unsupervised learning. 

**Objective:** To learn a compressed, efficient representation of input data without external labels.

Instead of predicting a target label $y$ from an input $x$, an autoencoder is trained to reconstruct its own input ($x \to x'$) after passing it through a constrained internal bottleneck.

An autoencoder consists of three main components:

1. **Encoder ($f_\theta$):** NN that compresses high-dimensional input data $x$ into a lower-dimensional latent representation $z$.

$$z = f_\theta(x)$$


2. **Bottleneck (Latent Space, $z$):** Low-dimensional layer in the network to force to ignore noise and preserve most important features of data.

3. **Decoder ($g_\phi$):** NN that takes the latent *code* $z$ and attempts to reconstruct the original input as $\hat{x}$.

$$\hat{x} = g_\phi(z)$$




<p style="page-break-after:always;"></p>

### Objective functions in training

NN learns to minimize a *reconstruction* loss ($\mathcal{L}$). 

The reconstruction loss measures the difference between the input $x$ and the reconstructed output $\hat{x}$. 

**Mean Squared Error (MSE)** 

<!--
Commonly used for continuous real-valued inputs (like images normalized between 0 and 1).

For a single sample with $D$ feature dimensions (e.g., $D$ pixels in an image), the total squared reconstruction loss is:
-->

$$\mathcal{L}_{\text{MSE}}(x, \hat{x}) = \frac{1}{D} \sum_{i=1}^{D} (x_i - \hat{x}_i)^2$$

where

* $x_i$ is the original value of feature $i$.
* $\hat{x}_i$ is the reconstructed value of feature $i$ (produced by the decoder).
* $D$ is the total number of input dimensions/features.

**Binary Cross-Entropy (BCE)** 

<!--
Often used when inputs are treated as Bernoulli distributions or binary values.

In autoencoders, Binary Cross-Entropy (BCE) measures the difference between the true input values $x$ and the reconstructed output values $\hat{x}$. It is typically used when the input data features are normalized to a continuous range of $[0, 1]$ (or are binary) and treated as probabilities. For a single sample with $D$ feature dimensions (e.g., $D$ pixels in an image), the Binary Cross-Entropy loss is:
-->

$$\mathcal{L}_{\text{BCE}}(x, \hat{x}) = -\sum_{i=1}^{D} \left[ x_i \log(\hat{x}_i) + (1 - x_i) \log(1 - \hat{x}_i) \right]$$

where

* $x_i \in [0, 1]$ is the original value of feature $i$.
* $\hat{x}_i \in (0, 1)$ is the reconstructed value of feature $i$ (usually produced by a sigmoid output layer).
* $D$ is the total number of input dimensions/features.



<p style="page-break-after:always;"></p>

### Variants

**Undercomplete AE**

Latent dimension is smaller than input dimension for basic dimensionality reduction and compression.

**[Denoising AE (DAE)](https://www.cs.toronto.edu/~larocheh/publications/icml-2008-denoising-autoencoders.pdf)**

Adds noise to inputs during training to prevent learning an identity mapping

The decoder is required to reconstruct the original, uncorrupted data learning to strip noise away

*Objective*: Noise removal and robust feature extraction

*Corruption:* An original input $x$ is stochastically corrupted using a corruption distribution $q(\tilde{x} \vert{} x)$ to produce a noisy input $\tilde{x}$. Common corruption strategies

* Additive Gaussian Noise: Adds random noise: $\tilde{x} = x + \epsilon$, where $\epsilon \sim \mathcal{N}(0, \sigma^2 I)$

* Masking: Randomly sets a fraction $\nu$ of input elements to 0 (or min/max values)

* Dropout Noise: Randomly zeros out features with probability $p$ during the forward pass

<p style="page-break-after:always;"></p>

**[Sparse AE](https://web.stanford.edu/class/cs294a/sparseAutoencoder_2011new.pdf)**

Forces hidden units to be mostly inactive (close to zero) via regularization penalties for any given input

*Objective*: Feature extraction without shrinking layer size

<!--
Unlike undercomplete autoencoders, a sparse autoencoder can actually have an overcomplete latent dimension (larger than the input dimension)

The sparsity constraint ensures that even with a large capacity, the network learns meaningful, disentangled representations rather than trivial identity mappings
-->

*Steps:*

* Forward pass: Input $x$ is passed through the encoder to obtain hidden activation vector $h = f_\theta(x)$

* Activation tracking: Across a mini-batch of N samples, the average activation $\hat{\rho}_j$ of each hidden neuron $j$ is calculated as

$$\hat{\rho}_j = \frac{1}{N} \sum_{i=1}^{N} a_j(x^{(i)})$$

where $a_j(x^{(i)})$ is the activation value of hidden unit $j$ when processing input $x^{(i)}$ (i.e. sigmoid in $(0, 1)$).


* Sparsity Penalty: A penalty term is added to the loss function that penalizes any hidden unit whose average activation deviates from a small target threshold $\rho$ (e.g., $\rho = 0.05$, meaning neurons should be active only 5% of the time)

$$\mathcal{L}_{\text{SAE}}(\theta, \phi) = \mathcal{L}(x, \hat{x}) + \beta \sum_{j=1}^{K} \text{Penalty}(\rho, \hat{\rho}_j) + \lambda \Omega(W)$$

where

* $\mathcal{L}$ is MSE or BCE loss function.
* $K$ is the total number of hidden units in the latent layer.
* $\rho$ is the target sparsity parameter (a value close to $0$).
* $\hat{\rho}_j$ is the empirical average activation of neuron $j$.
* $\beta$ is a hyperparameter controlling the weight of the sparsity penalty.
* $\lambda \Omega(W)$ is an optional weight decay regularizer ($L_2$ norm) to prevent overfitting.

*Sparsity penalty functions*

* Kullback-Leibler (KL) divergence: difference between two Bernoulli distributions: one with mean $\rho$ (target) and one with mean $\hat{\rho}_j$ (actual):

$$\text{KL}(\rho \parallel \hat{\rho}_j) = \rho \log \left( \frac{\rho}{\hat{\rho}_j} \right) + (1 - \rho) \log \left( \frac{1 - \rho}{1 - \hat{\rho}_j} \right)$$

* $L_1$ regularization penalty can be directly applied to the latent activations $h$:

$$\text{L1}(h) = \sum_{j=1}^{K} \vert{}h_j\vert{}$$

*Key Benefits*

* Feature interpretability: Each hidden unit specializes in detecting a distinct, highly specific subfeature

* Overcomplete capacities: Allows the latent dimension to be larger than the input dimension ($K > D$) without overfitting or degenerating into an identity copy operation.

* Modern LLM interpretability: Modern mechanistic interpretability uses massive sparse autoencoders to extract human-understandable concepts from the hidden activations of Transformer models.

<p style="page-break-after:always;"></p>

| Autoencoder Variant | Key Mechanism | Best Used For |
| --- | --- | --- |
| **Undercomplete AE** | Latent dimension is smaller than input dimension. | Basic dimensionality reduction & compression. |
| **Denoising AE (DAE)** | Adds noise to inputs during training; network learns to strip noise away. | Noise removal & robust feature extraction. |
| **Sparse AE** | Forces hidden units to be mostly inactive via regularization penalties. | Feature extraction without shrinking layer size. |
| **Variational AE (VAE)** | Encodes inputs into probability distributions (mean and variance) rather than fixed points. | Generative tasks (e.g., generating new realistic images). |
| **Convolutional AE (CAE)** | Replaces fully connected layers with convolutional and upsampling layers. | Processing image and spatial data. |

---

## 4. Key Applications

* **Dimensionality Reduction:** Serves as a non-linear alternative to Principal Component Analysis (PCA).
* **Anomaly & Outlier Detection:** If trained on normal data, the autoencoder will produce high reconstruction errors on unusual inputs.
* **Image Denoising:** Restores clean signals from corrupted inputs.
* **Pre-training & Representation Learning:** Learns rich representations from unlabeled data to initialize supervised models.

<p style="page-break-after:always;"></p>